In [ ]:
import pandas as pd
import requests
import time
import random
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
pd.set_option('display.max_colwidth', 200)
from datetime import date

In [5]:
# Scraping job_id API Jobstreet
def scrape_multiple_keywords(keywords, max_pages):
    all_data = []

    for keyword in keywords:
        print(f"Scraping jobs for keyword: {keyword}")
        job_id = []
        titles = []
        companies = []

        for page_number in range(1, max_pages + 1):
            api_url = f'https://id.jobstreet.com/api/jobsearch/v5/search?siteKey=ID-Main&sourcesystem=houston&userqueryid=45d2e71105d82100f674c0d5ac35d3cc-1751202&userid=89aba55b-b35f-40aa-9485-dfacb5738b52&usersessionid=89aba55b-b35f-40aa-9485-dfacb5738b52&eventCaptureSessionId=89aba55b-b35f-40aa-9485-dfacb5738b52&page={page_number}&keywords={keyword}&classification=6281&pageSize=32&include=seodata,relatedsearches,joracrosslink,gptTargeting,pills&baseKeywords=informatics&locale=id-ID&solId=d06fed81-7ac5-4601-9b9d-bfd4d8ae15c4&relatedSearchesCount=12'

            response = requests.get(api_url)
            if response.status_code == 200:
                data = response.json()

                for item in data['data']:
                    jid = item['id']
                    title = item['title']
                    company = item['advertiser'].get('description', '')

                    job_id.append(jid)
                    titles.append(title)
                    companies.append(company)
            else:
                print(f"Failed to retrieve data from the API. Status Code: {response.status_code}")
                break

        for jid, title, company in zip(job_id, titles, companies):
            all_data.append({
                'job_id': jid,
                'job_title': title,
                'company': company,
                'keyword': keyword,
                'job_source': 'Jobstreet'
            })

    return pd.DataFrame(all_data)

# Setup Selenium WebDriver
def setup_driver():
    chrome_driver_path = "chromedriver.exe"
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) "
                         "Chrome/122.0.0.0 Safari/537.36")

    service = Service(executable_path=chrome_driver_path)
    driver = webdriver.Chrome(service=service, options=options)
    return driver

# Scraping deskripsi pekerjaan dari halaman job detail
def scrape_with_selenium(job_id):
    url = f'https://id.jobstreet.com/id/job/{job_id}'
    driver = setup_driver()

    try:
        driver.get(url)
        time.sleep(random.uniform(5, 7))

        job_desc_ul_elements = driver.find_elements(By.XPATH, '//div[@data-automation="jobAdDetails"]//ul')
        job_desc = "".join([ul.text for ul in job_desc_ul_elements])
        print(f"Job {job_id} scraped")

        return job_desc
    
    except Exception as e:
        print("Gagal mengambil data pekerjaan:", e)
        return None

    finally:
        driver.quit()

# Scraping deskripsi dan menyimpan semua data
def scrape_and_store_text(df):
    descriptions = []
    for idx, row in df.iterrows():
        desc = scrape_with_selenium(row['job_id'])
        descriptions.append(desc)
        print(f"Desc {row['job_id']} stored")
    df['job_description'] = descriptions
    return df[['job_source', 'keyword', 'job_title', 'company', 'job_description']]

In [ ]:
keywords_cs = ['informatics', 'informatics+engineering', 'computer+science']
keywords_is = ['information+systems', 'accounting+information+systems', 'informatics+management']
keywords_all = keywords_cs + keywords_is
max_pages = 3

# Step 1: Ambil metadata pekerjaan
df_jobs = scrape_multiple_keywords(keywords_all, max_pages)

# Step 2: Ambil deskripsi pekerjaan per job_id
df_final = scrape_and_store_text(df_jobs)

# Step 3: Simpan ke file Excel terpisah
df_cs = df_final[df_final['keyword'].isin(keywords_cs)]
df_is = df_final[df_final['keyword'].isin(keywords_is)]

today = date.today()
df_cs.to_excel(f'jobstreet_CS_{today}.xlsx', index=False)
df_is.to_excel(f'jobstreet_IS_{today}.xlsx', index=False)

print("Selesai.")

Scraping jobs for keyword: informatics
Scraping jobs for keyword: informatics+engineering
Scraping jobs for keyword: computer+science
Scraping jobs for keyword: information+systems
Scraping jobs for keyword: accounting+information+systems
Scraping jobs for keyword: informatics+management
Job 85810912 scraped
Desc 85810912 stored
Job 85067059 scraped
Desc 85067059 stored
Job 85743788 scraped
Desc 85743788 stored
Job 85543777 scraped
Desc 85543777 stored
Job 85742982 scraped
Desc 85742982 stored
Job 85010727 scraped
Desc 85010727 stored
Job 85292361 scraped
Desc 85292361 stored
Job 85629208 scraped
Desc 85629208 stored
Job 85296638 scraped
Desc 85296638 stored
Job 85587272 scraped
Desc 85587272 stored
Job 85262054 scraped
Desc 85262054 stored
Job 85587333 scraped
Desc 85587333 stored
Job 85628763 scraped
Desc 85628763 stored
Job 85548538 scraped
Desc 85548538 stored
Job 85354733 scraped
Desc 85354733 stored
Job 85134611 scraped
Desc 85134611 stored
Job 85605770 scraped
Desc 85605770 stor

In [7]:
# print("Jumlah baris CS sebelum dihapus duplikat:", len(result_df))

# df_cleaned = result_df.drop_duplicates(subset=['job_id', 'job_title', 'descriptions'])
# df_cleaned = df_cleaned.dropna(subset=['job_id', 'job_title', 'descriptions'])

# print("Jumlah baris setelah dihapus duplikat:", len(df_cleaned))